# Exploration and Preparation of DailyDialog Dataset

In [4]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login
import re
import os

c:\Users\Tim\Projects\thesis_pub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
login=login(token=os.getenv("HF-TOKEN"))

In [6]:
df_dict = load_dataset("roskoN/dailydialog")
print(df_dict)

Using the latest cached version of the dataset since roskoN/dailydialog couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'full' at C:\Users\Tim\.cache\huggingface\datasets\roskoN___dailydialog\full\1.0.0\5214b2a66405abf87fd229e5c1007985501ffe3e (last modified on Tue Apr 14 22:04:37 2026).


DatasetDict({
    train: Dataset({
        features: ['id', 'acts', 'emotions', 'utterances'],
        num_rows: 11118
    })
    validation: Dataset({
        features: ['id', 'acts', 'emotions', 'utterances'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['id', 'acts', 'emotions', 'utterances'],
        num_rows: 1000
    })
})


In [7]:
train = df_dict["train"].to_pandas()
val = df_dict["validation"].to_pandas()
test = df_dict["test"].to_pandas()

df = pd.concat([train, val, test]).reset_index(drop=True)

In [8]:
print(df["utterances"][0])

['Say , Jim , how about going for a few beers after dinner ?'
 'You know that is tempting but is really not good for our fitness .'
 'What do you mean ? It will help us to relax .'
 "Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?"
 "I guess you are right.But what shall we do ? I don't feel like sitting at home ."
 'I suggest a walk over to the gym where we can play singsong and meet some of our friends .'
 "That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them ."
 'Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too .'
 "Good.Let ' s go now ." 'All right .']


In [9]:
def clean_utterance(text):
    text = text.strip()
    # delete space before punctuation
    text = re.sub(r'\s([?.!,;:\'])', r'\1', text)
    # make sure there is space after punctuation
    text = re.sub(r'([?.!])([A-Z])', r'\1 \2', text)
    return text

In [10]:
df_new = pd.DataFrame(columns = ["Person A", "Person B"])

for i in range(len(df)):
    dialog = df["utterances"][i] # putting dialog at index i into list
    person_a = " ".join([dialog[j] for j in range(0, len(dialog), 2)]) # putting every second line starting at index 0 into list
    person_b = " ".join([dialog[j] for j in range(1, len(dialog), 2)]) # putting every second line starting at index 1 into list
    data = {"Person A": [clean_utterance(person_a)], # putting those together into dictionary, using cleaning function
            "Person B": [clean_utterance(person_b)],
            "turns": [dialog]}

    temp = pd.DataFrame(data) # temporary df out of dict

    df_new = pd.concat([df_new, temp]).reset_index(drop = True) # computational not very efficient



In [11]:
df_new = df_new.loc[df_new["Person A"].str.len() > 100].reset_index(drop = True)
df_new = df_new.loc[df_new["Person B"].str.len() > 100].reset_index(drop = True)

df_new = df_new.drop_duplicates(subset=["Person A"]).reset_index(drop = True)
df_new = df_new.drop_duplicates(subset=["Person B"]).reset_index(drop = True)

In [ ]:
# dropped instances where situational descriptions are given for example "(Bob groans)" or "(Before christmas party)"
df_final = df_new.drop([7685, 6278, 1835, 5949, 5198, 4167, 2296, 2751]).reset_index(drop = True)

In [ ]:
#df_final.to_csv("..\data\csv\dailydialog_prep.csv", header = ["Person A", "Person B"], sep=",",index = False, encoding="utf-8")
#df_final.to_csv("..\data\csv\dailydialog_turns.csv", header = ["Person A", "Person B", "turns"], sep=",",index = False, encoding="utf-8")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_22864\600247383.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_final.to_csv("..\data\csv\dailydialog_turns.csv", header = ["Person A", "Person B", "turns"], sep=",",index = False, encoding="utf-8")
